# FashionSigLIP 다중 의류 속성 헤드 학습

FashionSigLIP 이미지 인코더는 고정하고, 의류 crop의 임베딩 위에 속성별 작은 분류 헤드만 학습합니다. 빈 라벨은 `없음`이 아니라 `미주석`으로 처리됩니다.

## 0. 팀원별 경로 설정

아래 경로만 각자 로컬 환경에 맞게 수정합니다. 기본값은 이 프로젝트에서 준비한 Fashionpedia + Fashion200K 통합 CSV와 이미지 구조입니다. 다른 데이터셋이나 직접 만든 CSV도 같은 형식이면 사용할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_DIR_INPUT = r''  # 비우면 현재 폴더 또는 하위 프로젝트를 자동 탐색
IMAGE_ROOT_INPUT = r'data'  # CSV의 image_path가 시작되는 공통 폴더
FASHIONPEDIA_TRAIN_JSON_INPUT = r''
FASHIONPEDIA_VAL_JSON_INPUT = r''
TRAIN_IMAGE_PREFIX = r''  # 예: train, 한 폴더에 이미지가 모두 있으면 비움
VAL_IMAGE_PREFIX = r''    # 예: val
ANNOTATION_CSV_INPUT = r'data/fashion_attribute_annotations.csv'
TRAIN_CACHE_INPUT = r'data/cache/fashion_attributes_train.pt'
VAL_CACHE_INPUT = r'data/cache/fashion_attributes_val.pt'
OUTPUT_CHECKPOINT_INPUT = r'models/fashion_attribute_heads.pt'

if PROJECT_DIR_INPUT:
    PROJECT_DIR = Path(PROJECT_DIR_INPUT).expanduser().resolve()
else:
    candidates = [Path.cwd(), Path.cwd() / 'ai_fashion_recommender']
    PROJECT_DIR = next((path.resolve() for path in candidates if (path / 'config.py').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('PROJECT_DIR_INPUT에 ai_fashion_recommender 폴더를 입력하세요.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def local_path(value):
    path = Path(value).expanduser()
    return (PROJECT_DIR / path).resolve() if value and not path.is_absolute() else path.resolve()

IMAGE_DIR = local_path(IMAGE_ROOT_INPUT) if IMAGE_ROOT_INPUT else ANNOTATION_CSV.parent
TRAIN_JSON = local_path(FASHIONPEDIA_TRAIN_JSON_INPUT) if FASHIONPEDIA_TRAIN_JSON_INPUT else None
VAL_JSON = local_path(FASHIONPEDIA_VAL_JSON_INPUT) if FASHIONPEDIA_VAL_JSON_INPUT else None
ANNOTATION_CSV = local_path(ANNOTATION_CSV_INPUT)
TRAIN_CACHE = local_path(TRAIN_CACHE_INPUT)
VAL_CACHE = local_path(VAL_CACHE_INPUT)
OUTPUT_CHECKPOINT = local_path(OUTPUT_CHECKPOINT_INPUT)
print('프로젝트:', PROJECT_DIR)
print('이미지:', IMAGE_DIR)
print('학습 CSV:', ANNOTATION_CSV)
print('출력 체크포인트:', OUTPUT_CHECKPOINT)

## 1. 속성 헤드 확인

카테고리·하의 종류·다리 모양·바지 전용 기장·소매·넥라인·칼라·핏은 단일 분류, 패턴·소재·디테일은 복수 선택으로 학습합니다. 현재는 17개 헤드와 총 124개 라벨입니다. 하의는 종류, 다리 모양, 기장, 구조 디테일을 별도 축으로 학습하므로 카고 팬츠이면서 와이드핏인 경우처럼 여러 특징을 함께 표현할 수 있습니다.

In [ ]:
from fashion_attribute_schema import ATTRIBUTE_TASKS

for name, task in ATTRIBUTE_TASKS.items():
    task_type = '다중 분류' if task.multi_label else '단일 분류'
    print(f'{name:15s} {task_type:7s} {len(task.labels):2d}개: {list(task.labels)}')

## 2. Fashionpedia JSON을 학습 CSV로 변환

공식 train/val JSON을 각각 변환한 뒤 한 CSV로 합칩니다. Fashionpedia만으로 구분하기 어려운 셔츠·블라우스·폴로 셔츠와 소재는 별도 라벨 데이터로 보충해야 합니다. 프로젝트의 통합 CSV를 사용하거나 이미 자체 CSV가 있으면 이 셀을 건너뜁니다.

In [ ]:
import csv
from fashion_attribute_dataset import convert_fashionpedia_instances

RUN_FASHIONPEDIA_CONVERSION = False
if RUN_FASHIONPEDIA_CONVERSION:
    if not TRAIN_JSON or not VAL_JSON:
        raise ValueError('0번 셀에 Fashionpedia train/val JSON 경로를 입력하세요.')
    train_csv = ANNOTATION_CSV.with_name('fashion_attributes_train.csv')
    val_csv = ANNOTATION_CSV.with_name('fashion_attributes_val.csv')
    print(convert_fashionpedia_instances(
        TRAIN_JSON, train_csv, split='train', image_prefix=TRAIN_IMAGE_PREFIX,
    ))
    print(convert_fashionpedia_instances(
        VAL_JSON, val_csv, split='val', image_prefix=VAL_IMAGE_PREFIX,
    ))
    ANNOTATION_CSV.parent.mkdir(parents=True, exist_ok=True)
    with ANNOTATION_CSV.open('w', encoding='utf-8-sig', newline='') as output:
        writer = None
        for source in (train_csv, val_csv):
            with source.open(encoding='utf-8-sig', newline='') as handle:
                reader = csv.DictReader(handle)
                if writer is None:
                    writer = csv.DictWriter(output, fieldnames=reader.fieldnames)
                    writer.writeheader()
                writer.writerows(reader)
    print('통합 CSV:', ANNOTATION_CSV)

## 3. 라벨 검증

이미지 존재 여부, 정의되지 않은 라벨, train/val 분리를 학습 전에 확인합니다. `pattern`, `material`, `detail`, `lower_detail`의 복수 라벨은 `|`로 구분합니다.

In [ ]:
from collections import Counter
from fashion_attribute_dataset import load_attribute_csv

if not ANNOTATION_CSV.is_file():
    raise FileNotFoundError(f'학습 CSV가 없습니다: {ANNOTATION_CSV}')
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'이미지 루트가 없습니다: {IMAGE_DIR}')
train_records = load_attribute_csv(ANNOTATION_CSV, IMAGE_DIR, split='train')
val_records = load_attribute_csv(ANNOTATION_CSV, IMAGE_DIR, split='val')
print('train:', len(train_records), 'val:', len(val_records))
for split_name, records in [('train', train_records), ('val', val_records)]:
    counts = Counter(task for record in records for task in record.labels)
    print(split_name, dict(counts))

print('\n[train 라벨 커버리지]')
for task_name, task in ATTRIBUTE_TASKS.items():
    label_counts = Counter(
        label for record in train_records for label in record.labels.get(task_name, [])
    )
    insufficient = {label: label_counts[label] for label in task.labels if label_counts[label] < 5}
    print(f'{task_name:15s} 주석 {sum(label_counts.values()):6d}개 / 5장 미만: {insufficient or "없음"}')

## 4. FashionSigLIP 임베딩 캐시 생성

이 단계에서만 무거운 FashionSigLIP을 실행합니다. 백본은 `eval` 상태이고 모든 파라미터의 gradient가 꺼집니다. 이후 epoch에서는 저장된 임베딩만 읽습니다. 배치마다 중간 파일을 저장하므로 중단 후 다시 실행하면 이어서 계산합니다. `reuse_cache_paths`에 기존 캐시를 넣으면 같은 이미지의 특징도 재사용할 수 있습니다. 첫 실행에는 모델 다운로드가 필요합니다.

In [ ]:
from config import FASHION_SIGLIP_MODEL_ID
from fashion_attribute_training import prepare_embedding_caches

RUN_EMBEDDING_CACHE = False
if RUN_EMBEDDING_CACHE:
    prepare_embedding_caches(
        ANNOTATION_CSV, IMAGE_DIR, TRAIN_CACHE, VAL_CACHE,
        model_id=FASHION_SIGLIP_MODEL_ID, device='auto', batch_size=64,
    )
    print('캐시 생성 완료:', TRAIN_CACHE, VAL_CACHE)

## 5. 속성별 분류 헤드 학습

17개 head만 학습합니다. 다중 라벨은 BCE loss와 양성 클래스 가중치, 단일 라벨은 완만한 역빈도 클래스 가중치를 적용한 cross entropy를 사용합니다. 각 헤드별로 validation loss가 가장 낮았던 시점을 따로 저장합니다. 다중 라벨 임계값과 하의 세부 단일 라벨의 신뢰도 임계값은 validation에서 조정합니다.

In [ ]:
from fashion_attribute_training import TrainingConfig, train_attribute_heads

RUN_TRAINING = False
if RUN_TRAINING:
    if not TRAIN_CACHE.is_file() or not VAL_CACHE.is_file():
        raise FileNotFoundError('4번 셀에서 train/val 임베딩 캐시를 먼저 만드세요.')
    summary = train_attribute_heads(
        TRAIN_CACHE, VAL_CACHE, OUTPUT_CHECKPOINT,
        config=TrainingConfig(
            epochs=50, batch_size=128, learning_rate=5e-4,
            hidden_dim=256, dropout=0.30, patience=8,
            minimum_label_examples=5,
        ),
        device='auto',
    )
    print(json.dumps(summary['metrics'], ensure_ascii=False, indent=2))

## 6. 평가 결과 확인

단일 속성은 accuracy, 다중 속성은 micro F1을 확인합니다. 표본이 너무 적거나 점수가 낮은 head는 `main.ipynb`에 연결하기 전에 라벨을 보강해야 합니다.

In [ ]:
metrics_path = OUTPUT_CHECKPOINT.with_suffix('.metrics.json')
if metrics_path.is_file():
    report = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('백본:', report['backbone_model_id'])
    print('표본:', report['train_samples'], '/', report['val_samples'])
    print(json.dumps(report['metrics'], ensure_ascii=False, indent=2))
else:
    print('아직 평가 결과가 없습니다:', metrics_path)

## 7. 메인 모델 연결

체크포인트가 `models/fashion_attribute_heads.pt`에 있으면 `main.ipynb`가 자동으로 불러옵니다. 다른 위치라면 메인 Notebook의 `ATTRIBUTE_HEADS_PATH_INPUT`을 변경합니다. 체크포인트가 없거나 분류가 임계값을 넘지 못하면 기존 FashionSigLIP 제로샷 및 마스크 측정값으로 자동 복귀합니다.